# Governance

AIDB agents can be **governed** by defining a *Purpose*. Within a purpose, you define the database *role* the agent operates as. This means an agent can only see and do what the database allows that role to see and do. The rule is enforced by the database, not by a prompt that an agent might ignore.

## The scenario

This demo builds two agents for searching credit card transactions:

| | Support Analyst agent | Supervisor agent |
|---|---|---|
| Database role | `cc_support_analyst` | `cc_supervisor` |
| Card number | Masked, showing only the last four digits | Full number |
| Cardholder personal data (name, address, birth date, and so on) | Hidden | Visible |
| Transaction details (merchant, category, amount, date) | Visible | Visible |

Both agents get the same instructions, the same model, and the same tools. The *only* difference is the purpose they are tied to.

## How the protection works

1. A **view** applies column-level security, returning masked or `NULL` values unless the current user is a member of the `cc_supervisor` role.
2. Each **purpose** maps to a database role.
3. Each **agent** is created with a purpose, so every query it runs executes as that role.
4. The roles can only read the **view**, never the underlying table.

## Claude API Key

This demo creates agents that use a Claude Sonnet model, so an Anthropic API key is required. Run the next cell and paste your key when prompted. It is held in memory only and passed to the database when the model is registered.

In [ ]:
# Provide your Claude API Key
import json
from getpass import getpass
claude_api_key = getpass("Claude API key: ")
creds = json.dumps({"api_key": claude_api_key})

## Connect to the database

Creates a SQLAlchemy engine for the database and attaches the `%sql` magics to it. `pool_pre_ping=True` checks connections before reuse, and named parameters are enabled so `%%sql` cells can use Python variables such as `:creds`.

Run this notebook as a user that can create roles and views. Later cells use `SET ROLE` to act as the demo roles.

In [ ]:
from sqlalchemy import create_engine
import os, pwd

PGUSER = pwd.getpwuid(os.geteuid()).pw_name
PGPORT="5432"
PGDATABASE = os.environ.get("PGDATABASE", "aidb")

if PGUSER == "postgres":
    PGSOCKET = "/run/postgresql"
else:
    PGSOCKET = "/tmp"

conn = create_engine(
    f"postgresql+psycopg2://{PGUSER}@/{PGDATABASE}?host={PGSOCKET}&port={PGPORT}"
)

%reload_ext sql
%sql conn
%config SqlMagic.named_parameters = "enabled"

## Create roles and schemas

This cell sets up the two database roles used for governance. It starts by removing anything left over from a previous run (the view, existing grants, schemas, and the roles), so it can be run repeatedly. It then:

- Creates `cc_support_analyst` and `cc_supervisor`.
- Grants both roles `USAGE` on the `aidb` schema and `EXECUTE` on the AIDB agent tool functions (`semantic_kb_search`, `search_by_comment`, `list_relationships`, `find_join_path`, and `run_sql_query`).
- Creates the `cc_demo` and `cc_raw` schemas.

The tool grants matter because an agent tied to one of these roles runs its tools *as that role*. Without them, the role could not use the tools at all. Notice that neither role is granted any access to the transactions table yet. That comes next, and only through a view.

In [ ]:
%%sql
DROP SCHEMA IF EXISTS cc_demo CASCADE;
DROP SCHEMA IF EXISTS cc_raw CASCADE;
CREATE SCHEMA cc_demo;
CREATE SCHEMA cc_raw;

-- Clean up grants from a previous run (skipped if the roles don't exist yet)
DO $$
DECLARE r text;
BEGIN
  FOREACH r IN ARRAY ARRAY['cc_support_analyst','cc_supervisor'] LOOP
    IF EXISTS (SELECT 1 FROM pg_roles WHERE rolname = r) THEN
      EXECUTE format('REVOKE ALL ON ALL FUNCTIONS IN SCHEMA aidb FROM %I', r);
      EXECUTE format('REVOKE USAGE ON SCHEMA aidb FROM %I', r);
    END IF;
  END LOOP;
END $$;

DROP ROLE IF EXISTS cc_support_analyst;
CREATE ROLE cc_support_analyst;

DROP ROLE IF EXISTS cc_supervisor;
CREATE ROLE cc_supervisor;

GRANT USAGE ON SCHEMA aidb TO cc_support_analyst;
GRANT USAGE ON SCHEMA cc_demo TO cc_support_analyst;
GRANT USAGE ON SCHEMA aidb TO cc_supervisor;
GRANT USAGE ON SCHEMA cc_demo TO cc_supervisor;

DO $$
DECLARE f regprocedure;
BEGIN
  FOR f IN
    SELECT p.oid::regprocedure
    FROM pg_proc p JOIN pg_namespace n ON n.oid = p.pronamespace
    WHERE n.nspname = 'aidb'
      AND p.proname IN ('semantic_kb_search','search_by_comment',
                        'list_relationships','find_join_path','run_sql_query')
  LOOP
    EXECUTE format('GRANT EXECUTE ON FUNCTION %s TO cc_support_analyst, cc_supervisor', f);
  END LOOP;
END $$;

## Sample data

This cell loads credit card transactions from the Hugging Face dataset `pointe77/credit-card-transaction`. It then:

1. Randomly samples 100,000 transactions to keep the demo fast.
2. Keeps 15 columns: the transaction (number, timestamp, merchant, category, amount), the card number, and personal details about the cardholder (name, gender, street, city, state, zip, job, and date of birth).
3. Renames `first` and `last` to `first_name` and `last_name`, and drops rows with no amount.
4. Writes the result to `cc_raw.credit_card_transactions`, dropping any previous copy first.

The cardholder columns are exactly the kind of sensitive data this demo protects. The first run downloads the dataset, so it may take a few minutes.

In [ ]:
from datasets import load_dataset
from sqlalchemy import text
import pandas as pd

# 1. Load the dataset
dataset = load_dataset("pointe77/credit-card-transaction")
df = dataset["train"].to_pandas()

# Randomly keep 100K rows
df = df.sample(n=100_000, random_state=42)

# 2. Pick and rename the columns you want to keep
columns_to_keep = [
    "trans_num",
    "trans_date_trans_time",
    "cc_num",
    "merchant",
    "category",
    "amt",
    "first",
    "last",
    "gender",
    "street",
    "city",
    "state",
    "zip",
    "job",
    "dob",
]

df_clean = df[[col for col in columns_to_keep if col in df.columns]].copy()
df_clean = df_clean.rename(columns={"first": "first_name", "last": "last_name"})

# 3. Drop any rows where the amount is null
df_clean = df_clean.dropna(subset=["amt"])

# 4. Load into the database
df_clean.to_sql(
    name="credit_card_transactions",
    con=conn,
    schema="cc_raw",
    if_exists="fail",
    index=False,
    chunksize=2000,
    method="multi",
)

print(f"Successfully imported {len(df_clean):,} transactions into cc_demo.credit_card_transactions!")

## Review data

Preview the first ten rows of the raw table. At this point nothing is protected: the full card number and all of the cardholder's personal details are visible to anyone who can query the table. The rest of the notebook fixes that.

In [ ]:
%%sql

SELECT *
FROM cc_raw.credit_card_transactions
LIMIT 10;

## Create a View with column level security

A view lets us control *which columns each role sees* without changing or copying the data.

The view `cc_demo.vw_credit_card_transactions` checks `pg_has_role(current_user, 'cc_supervisor', 'MEMBER')` for each sensitive column:

- **Supervisors** see the real values.
- **Everyone else** sees a masked card number (every digit replaced with `X` except the last four) and `NULL` for the cardholder's name, gender, address, job, and date of birth.
- Non-sensitive columns (transaction number, date, merchant, category, and amount) are visible to both roles.

Two other details are worth noting:

- `SET ROLE NONE` runs first so the view is created as your normal login user rather than as a demo role.
- `SELECT` is granted on the **view only**. The roles were never granted access to `credit_card_transactions` itself, so they cannot bypass the masking by querying the base table.

The view and each of its columns also get a `COMMENT`. These descriptions are what the semantic KB will index, which helps the agent understand what the columns mean.

In [ ]:
%%sql

SET ROLE NONE;

CREATE OR REPLACE VIEW cc_demo.vw_credit_card_transactions AS
SELECT 
    trans_num,
    trans_date_trans_time,
    CASE WHEN pg_has_role(current_user, 'cc_supervisor', 'MEMBER') THEN cc_num::varchar ELSE 
      (REPEAT('X', (LENGTH(cc_num::varchar)-4)) || REVERSE(SUBSTRING(REVERSE(cc_num::varchar), 1, 4))) END AS cc_num, 
    merchant,
    category,
    amt,
    CASE WHEN pg_has_role(current_user, 'cc_supervisor', 'MEMBER') THEN first_name ELSE NULL END AS first_name, 
    CASE WHEN pg_has_role(current_user, 'cc_supervisor', 'MEMBER') THEN last_name ELSE NULL END AS last_name,
    CASE WHEN pg_has_role(current_user, 'cc_supervisor', 'MEMBER') THEN gender ELSE NULL END AS gender,
    CASE WHEN pg_has_role(current_user, 'cc_supervisor', 'MEMBER') THEN street ELSE NULL END AS street,
    CASE WHEN pg_has_role(current_user, 'cc_supervisor', 'MEMBER') THEN city ELSE NULL END AS city,
    CASE WHEN pg_has_role(current_user, 'cc_supervisor', 'MEMBER') THEN state ELSE NULL END AS state,
    CASE WHEN pg_has_role(current_user, 'cc_supervisor', 'MEMBER') THEN zip ELSE NULL END AS zip,
    CASE WHEN pg_has_role(current_user, 'cc_supervisor', 'MEMBER') THEN job ELSE NULL END AS job,
    CASE WHEN pg_has_role(current_user, 'cc_supervisor', 'MEMBER') THEN dob ELSE NULL END AS dob
FROM cc_raw.credit_card_transactions;

GRANT USAGE ON SCHEMA cc_demo TO cc_support_analyst;
GRANT USAGE ON SCHEMA cc_demo TO cc_supervisor;

GRANT SELECT ON cc_demo.vw_credit_card_transactions TO cc_support_analyst;
GRANT SELECT ON cc_demo.vw_credit_card_transactions TO cc_supervisor;

COMMENT ON VIEW cc_demo.vw_credit_card_transactions IS 'Credit card transactions.';
COMMENT ON COLUMN cc_demo.vw_credit_card_transactions.trans_num IS 'Unique identifier for a transaction.';
COMMENT ON COLUMN cc_demo.vw_credit_card_transactions.trans_date_trans_time IS 'Date of credit card transaction.';
COMMENT ON COLUMN cc_demo.vw_credit_card_transactions.cc_num IS 'Credit Card Number';
COMMENT ON COLUMN cc_demo.vw_credit_card_transactions.merchant IS 'Merchant or vendor where the credit card transaction occurred.';
COMMENT ON COLUMN cc_demo.vw_credit_card_transactions.category IS 'The category in which the transaction occurred.';
COMMENT ON COLUMN cc_demo.vw_credit_card_transactions.amt IS 'The dollar amount of the transaction.';
COMMENT ON COLUMN cc_demo.vw_credit_card_transactions.first_name IS 'First name of the credit card owner.';
COMMENT ON COLUMN cc_demo.vw_credit_card_transactions.last_name IS 'Last name of the credit card owner.';
COMMENT ON COLUMN cc_demo.vw_credit_card_transactions.gender IS 'Gender of the credit card owner.';
COMMENT ON COLUMN cc_demo.vw_credit_card_transactions.street IS 'Street address of the credit card owner.';
COMMENT ON COLUMN cc_demo.vw_credit_card_transactions.city IS 'City address of the credit card owner.';
COMMENT ON COLUMN cc_demo.vw_credit_card_transactions.state IS 'State address of the credit card owner.';
COMMENT ON COLUMN cc_demo.vw_credit_card_transactions.zip IS 'Zip code address of the credit card owner.';
COMMENT ON COLUMN cc_demo.vw_credit_card_transactions.job IS 'Job title of the credit card owner.';
COMMENT ON COLUMN cc_demo.vw_credit_card_transactions.dob IS 'Date of birth of the credit card owner.';

## Query View as `cc_support_analyst`

`SET ROLE` makes the current session act as the given role. When the Support Analyst role is active, the personal information is hidden: the card number shows only its last four digits, and the name, address, and other personal columns come back as `NULL`.

In [ ]:
%%sql
SET ROLE cc_support_analyst;
SELECT *
FROM cc_demo.vw_credit_card_transactions
LIMIT 10;

## Query View as `cc_supervisor`

Now switch to the Supervisor role and run the identical query. When the Supervisor role is active, the personal information is visible. Same view, same SQL, different results, decided entirely by the database role.

In [ ]:
%%sql

SET ROLE cc_supervisor;

SELECT *
FROM cc_demo.vw_credit_card_transactions
LIMIT 10;


## Unset the role

`SET ROLE` lasts for the rest of the session, so return to your normal user with `SET ROLE NONE`. Without this, the next cells would run as the Supervisor and fail or behave unexpectedly when creating models and agents.

In [ ]:
%%sql
SET ROLE NONE;

## Register Claude Sonnet

`aidb.create_model` registers the Claude Sonnet 5 model inside the database under the name `claude_model_cc_transactions`. Both agents in this notebook use this model for reasoning and for writing their answers.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'claude_model_cc_transactions';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.models WHERE name = v_name) THEN
        PERFORM aidb.delete_model(v_name);
    END IF;

    PERFORM aidb.create_model(
        name        => v_name,
        provider    => 'anthropic_messages',
        config      => aidb.anthropic_messages_config(model => 'claude-sonnet-5'),
        credentials => CAST(:creds AS JSONB),
        replace_credentials => true
    );
END
$do$;

## Create the Semantic Knowledge Base

The semantic knowledge base indexes your schema (tables, views, columns, and their comments) so an agent can find the right objects from a plain-language question and then write SQL against them. It enables users to ask questions about relational tables and get answers from your data.

This KB indexes the `cc_demo` schema and uses `auto_processing => 'Live'` so that it stays current as the schema changes. It is built over the comments you added to the view in the previous step, which is why those comments are worth writing carefully.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'cc_transactions_kb';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.list_semantic_kbs() WHERE name = v_name) THEN
        PERFORM aidb.delete_semantic_kb(v_name);
    END IF;

    PERFORM aidb.create_semantic_kb(
        name    => v_name,
        model   => 'bge-small-en-v1.5-f16',
        schemas => ARRAY['cc_demo'],
        auto_processing => 'Live'
    );
END
$do$;

## Define Purposes

A **Purpose** is the link between an agent and a database role. When an agent is created with a purpose, everything the agent does in the database happens *as that role*. This protects your data!

Here we define two purposes:

- `cc_support_analyst_purpose` is tied to the `cc_support_analyst` role.
- `cc_supervisor_purpose` is tied to the `cc_supervisor` role.

The agent cannot do more than its role allows, regardless of what it is asked or how its instructions are worded.

In [ ]:
%%sql
DO $$
DECLARE
    p text;
BEGIN
    FOREACH p IN ARRAY ARRAY['cc_support_analyst_purpose', 'cc_supervisor_purpose']
    LOOP
        IF EXISTS (SELECT NULL FROM aidb.purpose_registry WHERE deleted_at IS NULL AND name = p) THEN
            RAISE INFO 'DELETE %', p;
            PERFORM aidb.delete_purpose(p);
        END IF;
    END LOOP;

    PERFORM aidb.create_purpose(
        name        => 'cc_support_analyst_purpose',
        role        => 'cc_support_analyst',
        description => 'Credit Card Support Analyst'
    );

    PERFORM aidb.create_purpose(
        name        => 'cc_supervisor_purpose',
        role        => 'cc_supervisor',
        description => 'Credit Card Supervisor'
    );
END
$$;


## Create two Agents

There is an agent for Support Analysts and one for Supervisors.

Read the two definitions side by side. The instructions, model, tools, and iteration limit are identical. Notice how each agent defines a `purpose`. That single parameter is the only difference, and it is what decides which role the agent operates as and, therefore, which data it can see.

The instructions point both agents at the view (`cc_demo.vw_credit_card_transactions`) and follow the same process as in the Semantic KB demo: search the KB, write one read-only `SELECT`, run it, and show the SQL.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'demo_agent_cc_support_analysts';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.agents WHERE name = v_name) THEN
        PERFORM aidb.delete_agent(v_name, force => true);
    END IF;

    PERFORM aidb.create_agent(
      name         => v_name,
      instructions => $$You answer questions about credit card transactions.
    All data lives in cc_demo.vw_credit_card_transactions. Ignore all other schemas and tables.
    The knowledge base is always kb_name => 'cc_transactions_kb'. Pass it on every semantic_kb_search, search_by_comment, list_relationships and find_join_path call.
    
    Process:
    1. Call semantic_kb_search to find the relevant tables and columns. Reuse names you already found earlier in this conversation instead of searching again.
    2. If the tables are not directly related, call find_join_path to get the join route.
    3. Write one read-only SELECT using only objects you found. Always schema-qualify tables, for example cc_demo.vw_credit_card_transactions.
    4. Run it with run_sql_query. If it fails on an unknown column, query information_schema.columns for that view. Do not guess.
    5. Answer briefly and show the SQL you ran.
    
    Never guess table or column names. If the schema does not contain what is needed, say so.$$,
      model        => 'claude_model_cc_transactions',
      tools        => ARRAY['semantic_kb_search','search_by_comment','list_relationships','find_join_path','run_sql_query'],
      max_iterations => 10,
      purpose      => 'cc_support_analyst_purpose'  --Defines what the Agent can access via the Purpose Role
    );

    v_name := 'demo_agent_cc_supervisors';
    IF EXISTS (SELECT NULL FROM aidb.agents WHERE name = v_name) THEN
        PERFORM aidb.delete_agent(v_name, force => true);
    END IF;

    PERFORM aidb.create_agent(
      name         => v_name,
      instructions => $$You answer questions about credit card transactions.
    All data lives in cc_demo.vw_credit_card_transactions. Ignore all other schemas and tables.
    The knowledge base is always kb_name => 'cc_transactions_kb'. Pass it on every semantic_kb_search, search_by_comment, list_relationships and find_join_path call.
    
    Process:
    1. Call semantic_kb_search to find the relevant tables and columns. Reuse names you already found earlier in this conversation instead of searching again.
    2. If the tables are not directly related, call find_join_path to get the join route.
    3. Write one read-only SELECT using only objects you found. Always schema-qualify tables, for example cc_demo.vw_credit_card_transactions.
    4. Run it with run_sql_query. If it fails on an unknown column, query information_schema.columns for that view. Do not guess.
    5. Answer briefly and show the SQL you ran.
    
    Never guess table or column names. If the schema does not contain what is needed, say so.$$,
      model        => 'claude_model_cc_transactions',
      tools        => ARRAY['semantic_kb_search','search_by_comment','list_relationships','find_join_path','run_sql_query'],
      max_iterations => 10,
      purpose      => 'cc_supervisor_purpose'  --Defines what the Agent can access via the Purpose Role
    );

END
$do$;

## Supervisor asks a question

The question, *"What are the top 10 states in terms of sum of credit card transactions?"*, needs the cardholder's `state`, which is a protected column. As a Supervisor, who has access to the data, the SQL query is created and executed, and you get a ranking of states.

In [ ]:
from sqlalchemy import text
from IPython.display import Markdown, display
import pandas as pd

def ask(agent, question, conversation_id=None):
    """Ask the agent, render the answer, return (message, conversation_id).
    Pass a conversation_id from an earlier call to ask a follow-up."""
    with conn.begin() as c:
        df = pd.read_sql(
            text("""SELECT * FROM aidb.agent_converse(
                        CAST(:agent AS text), CAST(:q AS text),
                        conversation_id => CAST(:cid AS text))"""),
            c, params={"agent": agent, "q": question, "cid": conversation_id})
    row = df.iloc[0]
    if row["error"]:
        display(Markdown(f"**Agent error:** {row['error']}"))
    else:
        display(Markdown(row["message"]))
    # On an error the function returns a NULL id, so keep the one passed in
    new_id = row["conversation_id"]
    return row["message"], (conversation_id if pd.isna(new_id) else new_id)

AGENT = "demo_agent_cc_supervisors"
_, conversation_id = ask(AGENT, "What are the top 10 states in terms of sum of credit card transactions?")

## Support Analyst asks a question

This is the same question, but asked of the Support Analyst agent. The data being requested includes protected columns, but because the agent operates as an Analyst, the data is protected: `state` is `NULL` for this role. Expect the agent to tell you that it cannot produce the ranking, rather than returning the data. The agent has not been told to hide anything. The database simply does not give it the values.

In [ ]:
AGENT = "demo_agent_cc_support_analysts"
_, conversation_id = ask(AGENT, "What are the top 10 states in terms of sum of credit card transactions?")

## Recap

You protected sensitive data with two cooperating layers:

1. **A view** that masks columns based on role membership.
2. **Purposes** that tie each agent to a database role.

Because enforcement happens in the database, it applies equally to people writing SQL, applications, and AI agents.